# ToothInstanceNet Kaggle FREE-GPU Benchmark

Research-only AlignerStudio benchmark. This notebook never modifies production code, the clinical pipeline, the fail-closed segmentation gate, adapters, or canonical STL files. It performs inference only, uses no external VLM/API, and does not provision paid cloud resources.

The notebook delegates the reproducible run and report generation to `run_benchmark.py`.

## 1. Notebook Parameters and Safety Guardrails

Set Kaggle paths, immutable canonical hashes, and research-only flags.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

RESEARCH_ONLY = True
INFERENCE_ONLY = True
ALLOW_EXTERNAL_VLM = False
ALLOW_PAID_CLOUD = False
PRODUCTION_ROOT = Path('/kaggle/working/production-must-not-be-used')
CANONICAL_SHA256 = {
    'upper.stl': '60aaafed87818b7cffd904056c4055748692bc280069af16325bff3b446e5a48',
    'lower.stl': 'dc4f8b0d4e8d1c21ab45ee0e69457eb575b869bcd895fbda36de9fcb4387037b',
}
assert RESEARCH_ONLY and INFERENCE_ONLY and not ALLOW_EXTERNAL_VLM and not ALLOW_PAID_CLOUD
print('Research-only guardrails enabled; production and paid cloud paths are disabled.')

## 2. GPU Environment Probe (Name, CUDA, PyTorch, VRAM)

Probe the free Kaggle GPU before installing or executing model code.

In [ ]:
import torch

gpu_metadata = {
    'available': torch.cuda.is_available(),
    'name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'cuda_version': torch.version.cuda,
    'pytorch_version': torch.__version__,
    'total_vram_bytes': int(torch.cuda.get_device_properties(0).total_memory) if torch.cuda.is_available() else None,
    'free_vram_bytes': int(torch.cuda.mem_get_info(0)[0]) if torch.cuda.is_available() else None,
    'capability': torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
}
print(json.dumps(gpu_metadata, indent=2))

## 3. Audited Model Metadata Load (Repo + Checkpoint Source of Truth)

The runner validates the audited commit and checkpoint directory before inference.

In [ ]:
REPO_URL = 'https://github.com/nnistelrooij/3dteethland'
AUDITED_COMMIT = '424252e3d94a1565c8c2090eb5bb456b76386b93'
CHECKPOINT_DIR = Path('/kaggle/input/toothinstancenet-checkpoints')
for name in ('align.ckpt', 'instseg_full.ckpt', 'landmarks_full.ckpt'):
    print(name, CHECKPOINT_DIR / name, (CHECKPOINT_DIR / name).exists())
print('Audited repository:', REPO_URL, AUDITED_COMMIT)

## 4. Minimal Dependency Installation for ToothInstanceNet

Kaggle's preinstalled PyTorch is reused when compatible; the research requirements file supplies the minimum Python dependencies.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/kaggle/working/AlignerStudio/research/benchmark/toothinstancenet/kaggle/requirements-kaggle.txt'], check=True)


## 5. Repository Checkout and CUDA Kernel Build Smoke Test

Clone exactly the audited commit and build the upstream CUDA extension. Any CUDA or kernel error is recorded as a blocker; the architecture is never changed.

In [ ]:
REPO_DIR = Path('/kaggle/working/3dteethland')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', AUDITED_COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-v', '-e', str(REPO_DIR)], check=True)
print(subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip())

## 6. Kaggle Input Intake and Canonical STL SHA-256 Verification

Attach a Kaggle Dataset containing `upper.stl` and `lower.stl` under `/kaggle/input/alignerstudio-real-case/`. The runner rejects hash mismatches before copying.

In [ ]:
INPUT_DIR = Path('/kaggle/input/alignerstudio-real-case')
for name, expected in CANONICAL_SHA256.items():
    path = INPUT_DIR / name
    actual = __import__('hashlib').sha256(path.read_bytes()).hexdigest()
    print(name, actual, actual == expected)
    assert actual == expected

## 7. Working-Copy Preparation (CASE_upper.stl, CASE_lower.stl)

Copy and rename only into `/kaggle/working/toothinstancenet-benchmark/inputs/CASE/`.

In [ ]:
WORK_DIR = Path('/kaggle/working/toothinstancenet-benchmark')
WORK_INPUT_DIR = WORK_DIR / 'inputs'
WORK_INPUT_DIR.mkdir(parents=True, exist_ok=True)
for name in ('upper.stl', 'lower.stl'):
    shutil.copy2(INPUT_DIR / name, WORK_INPUT_DIR / f'CASE_{name}')
print(sorted(str(path) for path in WORK_INPUT_DIR.glob('CASE_*.stl')))

## 8. Conditional Preprocessing/Conversion (Only If Strictly Required)

ToothInstanceNet explicitly accepts `.stl`, `.ply`, and `.obj`; therefore no conversion is required. This section intentionally performs no conversion.

In [ ]:
conversion_record = {'performed': False, 'reason': 'Official README lists stl, ply, obj extensions; working STL names satisfy STEM_* convention via CASE_*.'}
print(conversion_record)

## 9. Checkpoint Acquisition and Integrity Validation

Use a Kaggle Dataset containing the already verified checkpoints or copy them from an approved external research cache. The cell does not download unverified weights.

In [ ]:
EXPECTED_CHECKPOINTS = {
    'align.ckpt': '890d8e02ce9a83253c7c914048bbbc0f4bcbccdda405214ab493d5dcd4d3974a',
    'instseg_full.ckpt': '100c68a9b120402cc75539eff6347bd998bce8b1d4f01550638f471797d70803',
    'landmarks_full.ckpt': '9d4489439a7e9cab4d368b51de5f6debc41540abd8936a8605afebf1bd2daeff',
}
for name, expected in EXPECTED_CHECKPOINTS.items():
    path = CHECKPOINT_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    digest = __import__('hashlib').sha256(path.read_bytes()).hexdigest()
    print(name, path.stat().st_size, digest, digest == expected)

## 10. Inference-Only Execution on Upper/Lower Cases

Run only the upstream `instances` inference path. This is the primary tooth instance segmentation pipeline; landmark JSON is secondary output when the full model naturally emits it. The runner does not train and does not invoke any VLM/API.

In [ ]:
RUNNER = Path('/kaggle/working/AlignerStudio/research/benchmark/toothinstancenet/kaggle/run_benchmark.py')
result = subprocess.run([
    sys.executable, str(RUNNER), '--repo', str(REPO_DIR), '--input-dir', str(INPUT_DIR),
    '--checkpoints', str(CHECKPOINT_DIR), '--work-dir', str(WORK_DIR),
], text=True, capture_output=True)
print(result.stdout[-12000:])
print(result.stderr[-12000:])
if result.returncode not in (0, 2):
    raise RuntimeError(f'Unexpected harness failure: {result.returncode}')

## 11. Artifact Export (Instance JSON, Landmark JSON, Visualization Meshes)

The upstream command writes JSON and landmark files beside the working scans when supported. Raw upstream files remain in the working directory; no production artifact is created.

In [ ]:
for path in sorted(WORK_DIR.rglob('*')):
    if path.is_file() and path.suffix.lower() in {'.json', '.ply', '.obj', '.stl'}:
        print(path.relative_to(WORK_DIR))

## 12. Prediction Validation Suite (Instances, FDI, Sizes, Empty/Duplicate/Missing)

`run_benchmark.py` records instance counts, FDI labels when present, duplicate labels, raw schema, and correspondence status. Missing-tooth and size interpretation remain explicit research findings, not clinical conclusions.

## 13. Original Vertex Correspondence Proof (Sampled Points vs Original STL)

The audited source keeps the full loaded mesh vertex tensor, records internal downsample indices with `inplace=False`, interpolates clustered predictions back to that full tensor, and writes per-vertex `instances` and `labels`. The runner additionally checks both array cardinalities and emitted mesh topology. A length match alone is never treated as proof.

In [ ]:
report_path = WORK_DIR / 'benchmark-report.json'
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps(report.get('ORIGINAL_MESH_MAPPING'), indent=2))
else:
    print('No report: inference/build was blocked before output.')

## 14. Isolated Research Mapping Layer (Nearest/Registered Vertex Remap)

The runner is prepared to classify non-vertex-length outputs as sampled/non-vertex. A nearest-vertex remap may be added only inside this Kaggle working directory after the raw output schema exposes coordinates; no production mapping is created automatically.

## 15. Runtime + Peak VRAM Profiling and Compatibility Blocker Handling

The runner captures wall-clock runtime, allocated/reserved peak VRAM, subprocess stdout/stderr, and exact inference failure signatures. A CUDA/kernel failure leaves `PRODUCTION_READY` false and stops interpretation.

## 16. Benchmark Report Generation (benchmark-report.json, benchmark-summary.md)

The runner generates both files under the Kaggle working directory with the required report keys and `PRODUCTION_READY: false`.

In [ ]:
if report_path.exists():
    report = json.loads(report_path.read_text())
    required = ['MODEL', 'CHECKPOINTS', 'GPU', 'CUDA', 'INPUT_VERTICES', 'INPUT_FACES', 'PREDICTED_INSTANCES', 'FDI_LABELS', 'MISSING_TEETH', 'DUPLICATE_TEETH', 'LANDMARKS', 'ORIGINAL_MESH_MAPPING', 'RUNTIME_SECONDS', 'PEAK_VRAM', 'FAILURES', 'QUALITY_ASSESSMENT', 'PRODUCTION_READY']
    print('Missing report keys:', [key for key in required if key not in report])
    print(json.dumps({key: report.get(key) for key in required}, indent=2, default=str))

## 17. Post-Benchmark Verification (86 Python + 22 TypeScript Tests, Hash Recheck)

Run these commands only after the Kaggle benchmark completes. They are included for reproducibility; they do not change production code.

In [ ]:
print('Local verification commands:')
print('source services/api/.venv/bin/activate && pytest -q')
print('pnpm -r test')
print('sha256sum data/benchmark/real-case/upper.stl data/benchmark/real-case/lower.stl')

**Stop condition:** Any dependency, CUDA, kernel, checkpoint, or output-correspondence failure is recorded in `FAILURES`; no architecture workaround, CPU substitute, fixture, VLM/API, or production integration is permitted.